<a href="https://colab.research.google.com/github/d-noe/NLP_DH_PSL_Fall2025/blob/main/code/3_supervised/Hands_on_3_CanonChallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# *The Digital Librarian* v.2

This session's experiments are largely inspired by the articles: *Literary Canonicity and Algorithmic Fairness: The Effect of Author Gender on Classification Models* [(Lassen et al., 2023)](https://ceur-ws.org/Vol-3834/paper76.pdf); and *Operationalizing Canonicity: A Quantitative Study of French 19th and 20th Century Literature* [(Barré et al., 2023)](https://culturalanalytics.org/article/88113-operationalizing-canonicity-a-quantitative-study-of-french-19th-and-20th-century-literature).

## Canonicity Prediction and Fairness


<!-- ![Woman writing. Edouard Manet (c. 1883)](https://uploads7.wikiart.org/images/edouard-manet/woman-writing.jpg!Large.jpg)
<p align="right">
  <i>Woman writing</i>. Edouard Manet (c. 1883). Huile sur toile.
</p> -->

![Empirical Construction, Istanbul. Julie Mehretu (2003)](https://cdn.sanity.io/images/476nwnl9/production/6e85372d72734b07c7fc395fed0d6050cbdd5bb4-3000x2015.jpg)
<p align="right">
  <i>Empirical Construction, Istanbul</i>. Julie Mehretu (2003). Acrylic and ink on canvas.
</p>




# 🟢 Import libraries

In [5]:
# For deep learning
import torch
from torch.utils.data import DataLoader

# For (pre-trained) LMs
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

# For data handling
import pandas as pd
from datasets import load_dataset

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# For document representation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sentence_transformers import SentenceTransformer

# Fro general purposes
from tqdm import tqdm
import numpy as np
import random

# For visualisation
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
!wget https://raw.githubusercontent.com/d-noe/NLP_DH_PSL_Fall2025/refs/heads/main/code/scripts/helpers.py
from helpers import load_csv_from_github, load_dataset_from_github

--2025-11-10 12:46:52--  https://raw.githubusercontent.com/d-noe/NLP_DH_PSL_Fall2025/refs/heads/main/code/scripts/helpers.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3469 (3.4K) [text/plain]
Saving to: ‘helpers.py.1’

helpers.py.1        100%[===================>]   3.39K  --.-KB/s    in 0s      

2025-11-10 12:46:52 (28.9 MB/s) - ‘helpers.py.1’ saved [3469/3469]



# 🔵 Helpers


- Function to evaluate the model
- check submission format

In [10]:
# ======================================================================
# Some security checks before submission
# ======================================================================

def check_before_submission(
    df,
):
    if not "prediction" in df.columns:
      raise ValueError(f"The submitted df does not contain a 'prediction' column. Make sure to name it accordingly.")
    if not len(df)==5084:
      raise ValueError(f"The submitted df length ({len(df)}) does not match the length of the test set ({5084}).")

    print("✅ Submission DataFrame is good to go!")
    return True

def save_for_submission(
    df,
    group_name:str,
):
  # Never too sure: double-check
  submission_ok = check_before_submission(df)
  if submission_ok:
    try: # try to save with current group name
        df.to_csv(f'{group_name}.csv', mode='x')
        print(f"File saved at: {group_name}.csv")
    except FileExistsError: # if file already exist --> modify name and try again
        if type(group_name.split("-")[-1])==int:
            sub_id = int(group_name.split("-")[-1])
            next_sub_id = sub_id+1
        else:
            next_sub_id = 1
        modified_groupname = f"{group_name}-{next_sub_id}"
        save_for_submission(df, modified_groupname)
  else:
      raise ValueError(f"The submission format is not respected. Please modify your submission df.")

# 🟣 Load and explore the data

## Loading dataset

In [14]:
dataset = load_dataset_from_github("data/canon_challenge/dataset")
dataset

DatasetDict({
    train: Dataset({
        features: ['author', 'author_gender', 'book_title', 'publication_year', 'label', 'text'],
        num_rows: 6955
    })
    validation: Dataset({
        features: ['author', 'author_gender', 'book_title', 'publication_year', 'label', 'text'],
        num_rows: 5146
    })
    test: Dataset({
        features: ['author', 'author_gender', 'book_title', 'publication_year', 'label', 'text'],
        num_rows: 5084
    })
})

⚠️ Caution: This time we are working with excerpts from French-language novels (not in English anymore)! It might be worth keeping this in mind when implementing your classifier.

See for example:

In [27]:
print(dataset["train"]["text"][0])

Les uns la blâmèrent, les autres l’approuvèrent ; enfin. beaucoup secouèrent la tête en disant que le premier mari était mort au bout de trois mois, le second au bout de deux mois, le troisième au bout d’un mois, et que, pour ne pas faire mentir le calcul nécrologique, je mourrais probablement, moi, la première nuit de mes noces. Mais la personne sur laquelle le coup porta le plus violemment fut la pauvre Schimindra. Les bontés que j’avais eues pour elle lui avaient fait pendant quelque temps concevoir l’espoir de devenir ma femme. Dans un moment de désespoir, elle m’avoua jusqu’où avait été son ambition ; mais je lui fis promptement et facilement comprendre quelle supériorité avait la belle Vanly-Tching, veuve d’un docteur, veuve d’un mandarin, veuve d’un juge civil, sur elle, qui n’était veuve que d’un singe.


The **task** proposed in this notebook is to **predict the `label`** of this dataset which encodes if a literary piece is considered **part of the "literary canon" or not**.

See below some examples in our training data:

In [35]:
n_print = 10

rdm_ids = np.random.randint(0, len(dataset["train"]), n_print)

for rdm_i in rdm_ids:
  rdm_row = dataset['train'][int(rdm_i)]
  print("---------")
  print(f"LABEL    : {rdm_row['label']} ( = {dataset['train'].features['label'].int2str(rdm_row['label'])})")
  print(f"TEXT     : {rdm_row['text']}")
  print(f"METADATA : Author: {rdm_row['author']} | Author Gender: {rdm_row['author_gender']} (= {dataset['train'].features['author_gender'].int2str(rdm_row['author_gender'])}) | Book title: {rdm_row['book_title']} | Publication year: {rdm_row['publication_year']}")

---------
LABEL    : 0 ( = canon)
TEXT     : Le père Ortolan, méridional à tête fine, jurisconsulte de renom, était aussi poète à ses heures. Il avait publié les Enfantines et tout en jurant ne jamais écrire que pour le jeune âge, il ne dédaignait pas à l'endroit de ses vers l'approbation des grandes personnes. Aussi ses soirées, très suivies par les indigènes des quartiers savants, offraient-elles un agréable et original mélange de jolies femmes, de professeurs et d'avocats, de gens doctes et de poètes. C'est comme poète qu'on m'invitait. Parmi les jeunes et antiques célébrités que je vis passer là dans le brouillard d'or des premiers éblouissements, vint un soir Emile Ollivier.
METADATA : Author: Daudet Alphonse | Author Gender: 1 (= male) | Book title: Souvenirs d un homme de lettres. | Publication year: 1888
---------
LABEL    : 1 ( = non-canon)
TEXT     :  — Tu pourras très bien déjeuner demain avec le reste du poulet, dit-elle en le serrant dans le garde-manger, où il alla rejoin

ℹ️ Note: you can train your classifiers using the training set, and evaluate in (/optimize hyperparameters) using the validation split. But you cannot use the test set. Indeed, the `label` (and only the labels, no other metadata) are kept secret untill submission! See:

In [26]:
set(list(dataset["test"]["label"])) # only 'None'!

{None}

## Explore the data

Ussually, before diving into the training part, it can be great to familiarize a bit with the data. This can sometimes even inform the design of the ensuing classifier, or warn on potential difficulties that may arise...

In [ ]:
# Tip: if more comfortable with pandas, you can convert each split of the data to pandas DataFrame with, e.g.:
# df_train = dataset["train"].to_pandas()

# 🟠 Implement your classifier

Now it is time to devise your classifier! You can be creative or conservative here. Try to think for instance on the way to use the data, the type of algorithm that you want to use, etc. (-> no bad answers here!).

Start by implementing the architecture and training components of your classifier. Train and validate it on the `"train"` and `"validation"` splits, and finally get the predictions on the `"test"` set to submit your results.

Feel free to re-use code from the [companion tutorial](https://github.com/d-noe/NLP_DH_PSL_Fall2025/blob/main/code/3_supervised/Tutorial_3_SFT.ipynb): be it fine-tuning example or obtaining documents representations (BoW, TF-IDF, dense embeddings, ...) and classifying them in the representation space.

## Train/Validate your classifier

## Run your classifier on the test set

# 🟡 Upload your Submission!


Your time to shine: store your predictions on the test set in a `pandas.DataFrame` into a column named `"prediction"`.

Check that your DataFrame complies with the necessary format for submission, if not, please modify it so that it respects the guidelines.

If it is compliant the format, then you can save the prediction DataFrame into `csv` format, named with the name of your submission, and uploaed it on the leaderboard submission platform at: https://leaderboard-performance-fairness.streamlit.app/.

In [ ]:
# # TODO: uncomment the following lines and store your predictions in a pandas DataFrame

# df_submission = pd.DataFrame()
# df_submission["prediction"] = [] # YOUR PREDICTIONS HERE!

In [13]:
check_before_submission(df_submission)

ValueError: The submitted df length (100) does not match the length of the test set (5084).

If you passed the quick format check: great! You can now save your predictions and upload it to the leaderboard to see how you did on the *hidden* test set!

In [11]:
save_for_submission(
    df_submission,
    group_name=, # YOUR GROUP NAME HERE
)

✅ Submission DataFrame is good to go!
File saved at: trial_submission.csv
